<a href="https://colab.research.google.com/github/yejikwon7/DeepLearning/blob/main/final2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets, transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset, TensorDataset
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

# YOLOv5 모델 불러오기 (PyTorch Hub)
yolo_model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

import sys
sys.path.append('/content/yolov5')  # YOLOv5 경로 추가

Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2024-12-21 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


In [ ]:
from models.common import DetectMultiBackend
from utils.general import non_max_suppression
from utils.torch_utils import select_device

In [ ]:
# GPU 메모리 관리
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
# !pip install kaggle
# from google.colab import files
# files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("sautkin/imagenet1k1")

# print("Path to dataset files:", path)

In [ ]:
# hyper-parameters
lr = 0.001
epochs = 30
batch_size = 16

In [ ]:
# 데이터셋 경로 설정
train_dir = "/root/.cache/kagglehub/datasets/sautkin/imagenet1k1/versions/2"
val_dir = "/root/.cache/kagglehub/datasets/sautkin/imagenet1k1/versions/2"

# 이미지 경로 수집 함수
def collect_image_paths(directory):
    image_paths = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):  # 이미지 파일 필터링
                image_paths.append(os.path.join(root, file))
    return image_paths

In [ ]:
# 이미지 경로 로드
train_image_paths = collect_image_paths(train_dir)
val_image_paths = collect_image_paths(val_dir)

# 데이터 로드 확인
if not train_image_paths or not val_image_paths:
    raise FileNotFoundError("Train or validation directory contains no images.")

In [ ]:
# 1000개의 데이터를 랜덤하게 샘플링
random.seed(42)
total_image_paths = train_image_paths + val_image_paths
sampled_image_paths = random.sample(total_image_paths, 1000)

In [ ]:
# # 데이터셋 로드 및 전처리
# train_image_paths = transforms.Compose([
#     transforms.Resize((640, 640)),  # YOLOv5가 요구하는 입력 크기로 조정
#     transforms.ToTensor(),          # 이미지를 Tensor로 변환
# ])

# val_image_paths = transforms.Compose([
#     transforms.Resize((640, 640)),  # YOLOv5가 요구하는 입력 크기로 조정
#     transforms.ToTensor(),          # 이미지를 Tensor로 변환
# ])

In [ ]:
# 데이터셋 정의
class CustomDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label
print(len(sampled_image_paths))

1000


In [ ]:
# 데이터셋 분리
train_size = int(0.6 * len(sampled_image_paths))
val_size = int(0.2 * len(sampled_image_paths))
test_size = len(sampled_image_paths) - train_size - val_size

train_image_paths = sampled_image_paths[:train_size]
val_image_paths = sampled_image_paths[train_size:train_size + val_size]
test_image_paths = sampled_image_paths[train_size + val_size:]

train_labels = [0] * len(train_image_paths)
val_labels = [0] * len(val_image_paths)
test_labels = [0] * len(test_image_paths)

train_transforms = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transforms = val_transforms

train_dataset = CustomDataset(train_image_paths, train_labels, transform=train_transforms)
val_dataset = CustomDataset(val_image_paths, val_labels, transform=val_transforms)
test_dataset = CustomDataset(test_image_paths, test_labels, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

In [ ]:
# YOLO 객체 검출
def extract_objects_with_yolo(yolo_model, dataset):
    cropped_images, labels = [], []
    for img, label in tqdm(dataset):
        img_pil = transforms.ToPILImage()(img)  # 수정: GPU 텐서라면 PIL 이미지로 변환
        results = yolo_model(img_pil)
        detections = results.xyxy[0].cpu().numpy()

        for det in detections:
            x1, y1, x2, y2, conf, cls = det
            cropped_images.append(img_pil.crop((x1, y1, x2, y2)))
            labels.append(int(cls))
    return cropped_images, labels

In [ ]:
# YOLO 객체 검출 실행
cropped_images, yolo_labels = extract_objects_with_yolo(yolo_model, train_dataset)

# 라벨 정규화 후 데이터 타입 확인 및 변환
yolo_labels = [label - min(yolo_labels) for label in yolo_labels]
yolo_labels = torch.tensor(yolo_labels, dtype=torch.long)

print(f"Unique labels: {torch.unique(yolo_labels)}")
print(f"Number of classes: {len(set(yolo_labels))}")

if len(cropped_images) == 0 or len(yolo_labels) == 0:
    raise ValueError("No objects detected. Cannot proceed with ResNet training.")

  0%|          | 0/600 [00:00<?, ?it/s]/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:892: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
WARNING ⚠️ NMS time limit 0.550s exceeded
  0%|          | 1/600 [00:00<08:46,  1.14it/s]/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:892: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:892: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
  0%|          | 3/600 [00:00<02:38,  3.77it/s]/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:892: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `to

Unique labels: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 79])
Number of classes: 1147


In [ ]:
# ResNet 학습 데이터 생성
resnet_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

resnet_inputs = [resnet_transforms(img) for img in cropped_images]
resnet_dataset = TensorDataset(torch.stack(resnet_inputs), torch.tensor(yolo_labels))
resnet_train_loader = DataLoader(resnet_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

In [ ]:
# ResNet 모델 정의
resnet_model = models.resnet50(pretrained=True)
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, len(set(yolo_labels)))

# ResNet 모델 출력 클래스 수정
num_classes = len(set(yolo_labels))
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

# 데이터와 모델 GPU로 전송 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet_model = resnet_model.to(device)

In [ ]:
# for inputs, labels in train_loader:
#     inputs, labels = inputs.to(device), labels.to(device)
#     print(f"Input shape: {inputs.shape}, Label shape: {labels.shape}")
#     print(f"Label values: {labels.unique()}")
#     break

In [ ]:
class DeviceDataLoader:
    def __init__(self, dataloader, device):
        self.dataloader = dataloader
        self.device = device
        self.dataset = dataloader.dataset  # 원래 DataLoader의 dataset 속성 유지

    def __iter__(self):
        for batch in self.dataloader:
            yield (item.to(self.device) for item in batch)

    def __len__(self):
        return len(self.dataloader)

In [ ]:
# 손실 함수 및 옵티마이저 정의
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet_model.parameters(), lr=lr)

In [ ]:
# ResNet 학습 함수
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_losses = []
    val_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in tqdm(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            # 데이터 타입 확인
            if labels.dtype != torch.long:
                print(f"Label dtype mismatch: {labels.dtype}. Converting to torch.long.")
                labels = labels.long()

            optimizer.zero_grad()
            outputs = model(inputs)  # 모델의 순전파 결과

            # 디버깅 출력
            print(f"Outputs shape: {outputs.shape}, Labels shape: {labels.shape}")
            print(f"Labels unique: {labels.unique()}")

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)

        # Validation
        model.eval()
        val_loss = 0.0
        corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                corrects += torch.sum(preds == labels.data)

        val_loss = val_loss / len(val_loader.dataset)
        accuracy = corrects.double() / len(val_loader.dataset)

        val_losses.append(val_loss)
        val_accuracies.append(accuracy.item())

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, Accuracy: {accuracy:.4f}")

    return train_losses, val_losses, val_accuracies
for inputs, labels in resnet_train_loader:
    print(inputs.device, labels.device)  # GPU로 전송 여부 확인

print(f"Model output size: {resnet_model.fc.out_features}")
print(f"Number of classes: {len(set(yolo_labels))}")

cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
cpu cpu
Model output size: 1147
Number of classes: 1147


In [ ]:
# 학습 실행
train_losses, val_losses, val_accuracies = train_model(resnet_model, resnet_train_loader, val_loader, optimizer, criterion, epochs=epochs)

  0%|          | 0/72 [00:00<?, ?it/s]

Outputs shape: torch.Size([16, 1147]), Labels shape: torch.Size([16])
Labels unique: tensor([ 0,  8, 27, 39, 46, 49, 65, 74, 79], device='cuda:0')


  3%|▎         | 2/72 [00:01<00:44,  1.57it/s]

Outputs shape: torch.Size([16, 1147]), Labels shape: torch.Size([16])
Labels unique: tensor([ 0,  1,  2,  7,  8, 25, 35, 45, 55, 73, 75], device='cuda:0')


  4%|▍         | 3/72 [00:01<00:30,  2.23it/s]

...

Outputs shape: torch.Size([16, 1147]), Labels shape: torch.Size([16])
Labels unique: tensor([ 0,  2, 16, 27, 39, 43, 49, 65, 75], device='cuda:0')


100%|██████████| 72/72 [00:15<00:00,  4.63it/s]

Outputs shape: torch.Size([11, 1147]), Labels shape: torch.Size([11])
Labels unique: tensor([ 0,  1,  7, 25, 43, 74], device='cuda:0')


100%|██████████| 72/72 [00:16<00:00,  4.48it/s]


Epoch 30/30, Train Loss: 10.7843, Val Loss: 3.0560, Accuracy: 0.3650


In [ ]:
# 결과 시각화

%matplotlib inline
plt.show(block=True)

def plot_metrics(train_losses, val_losses, val_accuracies):
    # 디버깅: 데이터 확인
    print("Train Losses:", train_losses)
    print("Validation Losses:", val_losses)
    print("Validation Accuracies:", val_accuracies)

    plt.figure(figsize=(12, 4))

    # Loss 그래프
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.legend()
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')

    # Accuracy 그래프
    plt.subplot(1, 2, 2)
    plt.plot(val_accuracies, label='Val Accuracy')
    plt.legend()
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')

    # 그래프 저장 및 출력
    plt.savefig("metrics_output.png")  # 그래프를 파일로 저장
    plt.show(block=True)  # 그래프 강제 렌더링

Train Losses: [59.86821931931708, 49.4319429828061, 48.224636448754204, 46.24652700622877, 44.17563800679313, 42.75327223208215, 42.162037620941796, 40.18406713505586, 39.8792335026794, 38.72506409883499, 38.30673287974464, 36.45477780203024, 35.72010820110639, 33.68645356761085, 33.14275734788842, 31.854748629861408, 29.528734248545433, 29.15545074476136, 27.396445588933098, 25.567929261260563, 24.858473300933838, 23.000252248512375, 21.984101477596496, 19.19149349961016, 19.03345341152615, 17.549106121891075, 15.3715427832471, 13.047249227762222, 11.017338946461678, 10.784296989440918]
Validation Losses: [1.7617602252960205, 1.3913844633102417, 1.3275594091415406, 2.0589821338653564, 1.6878539323806763, 2.0592912340164187, 1.4687010622024537, 2.2181870698928834, 1.6656766080856322, 1.3509368896484375, 2.6270471096038817, 5.002838678359986, 1.676169629096985, 2.427444267272949, 7.798927536010742, 2.5983037757873535, 1.337863438129425, 3.254966926574707, 3.615215940475464, 2.4902082967

In [ ]:
# Testing
def test_model(model, test_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        # 디버깅: 데이터 출력
        print("Input Tensor Shape:", inputs[0].shape)
        print("Predicted Label:", preds[0].item())
        print("True Label:", labels[0].item())

        # 이미지 변환 및 출력
        try:
            # Normalize 되지 않은 이미지를 위한 역변환
            unnormalize = transforms.Normalize(
                mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
                std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
            )
            img = unnormalize(inputs[0].cpu())  # Normalize 해제
            img = transforms.ToPILImage()(img)  # PIL 이미지 변환
            plt.imshow(img)
            plt.title(f"True: {labels[0].item()}, Predicted: {preds[0].item()}")
            plt.show(block=True)  # 강제 출력
        except Exception as e:
            print("Error displaying image:", e)
            continue

In [ ]:
# # Main execution
# cropped_images, yolo_labels = extract_objects_with_yolo(yolo_model, CustomDataset(...))  # 수정: CustomDataset을 생성 후 사용
# resnet_transforms = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])
# resnet_inputs = [resnet_transforms(img) for img in cropped_images]
# resnet_dataset = TensorDataset(torch.stack(resnet_inputs), torch.tensor(yolo_labels))
# resnet_train_loader = DataLoader(resnet_dataset, batch_size=16, shuffle=True)

# resnet_model = models.resnet50(pretrained=True)
# resnet_model.fc = nn.Linear(resnet_model.fc.in_features, len(set(yolo_labels)))

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(resnet_model.parameters(), lr=0.001)

# # 수정: train_loader 및 val_loader에 DeviceDataLoader 사용 (생략 가능)
# train_losses, val_losses, val_accuracies = train_model(resnet_model, resnet_train_loader, val_loader, optimizer, criterion, epochs=10)
# plot_metrics(train_losses, val_losses, val_accuracies)